# 4. Zone entries outcomes

Compare **carried and played 5v5 entries** by preceding forward possession pace and the percentage producing a shot attempt within **five seconds**.

Quartiles share the same thresholds across entry types.

"Does faster forward pace before zone entry coincide with more shot generation?"

## 4.1. Load inputs

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROCESSED_DATA_DIR = Path("../data/processed")
OUTCOME_SECONDS = 5
ENTRY_TYPES = ["Carried", "Played"]

# Event IDs preserve order, including events sharing a clock value.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
movements["distance_n_ft"] = movements["dx_ft"].clip(lower=0)
entries = events.loc[
    events["event"].eq("Zone Entry")
    & events["detail_1"].isin(ENTRY_TYPES)
    & events["is_5v5"]
].copy()
entries["entry_type"] = entries["detail_1"]

## 4.2. Calculate preceding forward possession pace

Pool movements in the entry's possession whose end event IDs are **up to and including** the entry.

Forward pace is summed positive attacking-direction distance divided by summed modeled movement time. Lateral and backward movements contribute time but no forward distance.

In [25]:
def add_preceding_pace(entry_events, transition_events):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for entry in entry_events.itertuples(index=False):
        history = movement_groups.get(entry.possession_id, transition_events.iloc[:0])
        history = history.loc[history["end_event_id"].le(entry.event_id)]
        duration = float(history["modeled_elapsed_seconds"].sum())
        forward_distance = float(history["distance_n_ft"].sum())
        rows.append(
            {
                "event_id": entry.event_id,
                "preceding_transitions": len(history),
                "preceding_movement_seconds": duration,
                "preceding_forward_distance_ft": forward_distance,
                "preceding_forward_pace_ft_s": forward_distance / duration
                if duration > 0
                else np.nan,
            }
        )
    return entry_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


entries = add_preceding_pace(entries, movements)

## 4.3. Identify shot outcomes

A subsequent entering-team `Shot` or `Goal` within five modeled seconds counts as success, including blocked and missed shots.

Require five seconds of **observed period follow-up**, measured against the last recorded modeled clock.

In [26]:
CONTROL_EVENTS = {
    "Puck Recovery",
    "Takeaway",
    "Faceoff Win",
    "Pass",
    "Pass Reception",
    "Incomplete Pass",
    "Shot",
    "Goal",
    "Dump In/Out",
    "Zone Entry",
}
RESTART_EVENTS = {"Faceoff Win", "Penalty Shot Goal", "Zone Entry"}


def add_shot_outcomes(entry_events, all_events):
    period_events = {
        key: group.sort_values("event_id")
        for key, group in all_events.groupby(["game_id", "period"], sort=False)
    }
    rows = []
    for entry in entry_events.itertuples(index=False):
        period = period_events[(entry.game_id, entry.period)]
        complete = (
            entry.modeled_clock_seconds - period["modeled_clock_seconds"].min()
            >= OUTCOME_SECONDS
        )
        following = period.loc[
            period["event_id"].gt(entry.event_id)
            & period["modeled_clock_seconds"].between(
                entry.modeled_clock_seconds - OUTCOME_SECONDS,
                entry.modeled_clock_seconds,
            )
        ]
        shot_event_id = None
        for event in following.itertuples(index=False):
            if not event.is_5v5 or event.event in RESTART_EVENTS:
                break
            if event.event in CONTROL_EVENTS and event.team_id != entry.team_id:
                break
            if event.event in {"Shot", "Goal"} and event.team_id == entry.team_id:
                shot_event_id = event.event_id
                break
        rows.append(
            {
                "event_id": entry.event_id,
                "complete_followup": bool(complete),
                "shot_event_id": shot_event_id,
                "shot_success": float(shot_event_id is not None)
                if complete
                else np.nan,
            }
        )
    return entry_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


entries = add_shot_outcomes(entries, events)

## 4.4. Summarize quartiles

In [27]:
def summarize_entries(data):
    pace_column = "preceding_forward_pace_ft_s"
    eligible = data.loc[data[pace_column].notna()].copy()
    bands, edges = pd.qcut(
        eligible[pace_column], q=4, labels=False, retbins=True, duplicates="drop"
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    analysis = eligible.loc[
        eligible["complete_followup"] & eligible["quartile"].notna()
    ]
    summary = (
        analysis.groupby(["entry_type", "quartile"], observed=True)
        .agg(
            entries=("event_id", "size"),
            successes=("shot_success", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=(pace_column, "median"),
        )
        .reset_index()
    )
    summary["shot_pct"] = 100 * summary["successes"] / summary["entries"]
    summary = summary.merge(boundaries, on="quartile", validate="many_to_one")

    # Coverage counts remain visible even when pace or follow-up is missing.
    coverage = (
        data.assign(
            missing_pace=data[pace_column].isna(),
            incomplete_followup=~data["complete_followup"],
            charted=data["event_id"].isin(analysis["event_id"]),
        )
        .groupby("entry_type")
        .agg(
            entries=("event_id", "size"),
            missing_pace=("missing_pace", "sum"),
            incomplete_followup=("incomplete_followup", "sum"),
            charted=("charted", "sum"),
        )
        .reset_index()
    )
    return summary, boundaries, coverage


entry_summary, quartile_boundaries, coverage_df = summarize_entries(entries)
display(coverage_df)

,entry_type,entries,missing_pace,incomplete_followup,charted
0,Carried,2120,16,7,2098
1,Played,256,0,0,256


## 4.5. Display entry outcomes

In [28]:
QUARTILE_COLORS = ["#bfdbfe", "#93c5fd", "#3b82f6", "#1d4ed8"]


def entry_outcomes_figure(summary, boundaries):
    figure = make_subplots(
        rows=1, cols=2, subplot_titles=ENTRY_TYPES, shared_yaxes=True
    )
    band_count = len(boundaries)
    for column, entry_type in enumerate(ENTRY_TYPES, start=1):
        # Missing groups appear as gaps, never zero-percent results.
        bars = summary.loc[summary["entry_type"].eq(entry_type)].set_index("quartile")
        bars = bars.reindex(range(1, band_count + 1))
        figure.add_trace(
            go.Bar(
                x=[f"Q{q}" for q in bars.index],
                y=bars["shot_pct"],
                marker_color=QUARTILE_COLORS[:band_count],
                showlegend=False,
                text=[
                    f"{value:.1f}%" if pd.notna(value) else ""
                    for value in bars["shot_pct"]
                ],
                textposition="outside",
                cliponaxis=False,
                customdata=bars[
                    [
                        "entries",
                        "successes",
                        "games",
                        "lower_pace_ft_s",
                        "upper_pace_ft_s",
                        "median_pace_ft_s",
                    ]
                ].to_numpy(),
                hovertemplate=(
                    "%{x}<br>Shot-producing entries: %{y:.1f}%"
                    "<br>Entries: %{customdata[0]:.0f}"
                    "<br>Successful entries: %{customdata[1]:.0f}"
                    "<br>Games: %{customdata[2]:.0f}"
                    "<br>Pace range: %{customdata[3]:.2f}"
                    " to %{customdata[4]:.2f} ft/s"
                    "<br>Median pace: %{customdata[5]:.1f} ft/s<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
    if summary.empty:
        note = "No data: no assigned pace bands with complete follow-up."
    elif band_count < 4:
        note = f"Only {band_count} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title=(
            f"Forward pace before entry | 5v5 | shot within {OUTCOME_SECONDS} seconds"
        ),
        template="plotly_white",
        height=500,
        margin=dict(t=100, b=100),
    )
    figure.update_xaxes(
        title_text="Preceding forward possession pace",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    figure.update_yaxes(
        range=[0, 100], title_text="Shot-producing entries (%)", row=1, col=1
    )
    figure.update_yaxes(range=[0, 100], row=1, col=2)
    return figure


forward_pace_figure = entry_outcomes_figure(entry_summary, quartile_boundaries)
forward_pace_figure.show()

## 4.6. Export entry outcomes

In [29]:
# Keep the final table ordered and ready for direct database ingestion.
entry_outcomes_df = (
    entry_summary[
        [
            "entry_type",
            "quartile",
            "entries",
            "successes",
            "games",
            "median_pace_ft_s",
            "shot_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "entry_type": "string",
            "quartile": "Int64",
            "entries": "Int64",
            "successes": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "shot_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values(["entry_type", "quartile"])
    .reset_index(drop=True)
)

export_path = PROCESSED_DATA_DIR / "entry_outcomes.parquet"
entry_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)

# Verify the saved values and types before reporting success.
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, entry_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 8 rows and 9 columns to ..\data\processed\entry_outcomes.parquet


,entry_type,quartile,entries,successes,games,median_pace_ft_s,shot_pct,lower_pace_ft_s,upper_pace_ft_s
0,Carried,1,531,224,34,8.0,42.184557,0.0,10.487245
1,Carried,2,537,239,34,12.428571,44.506518,10.487245,14.773504
2,Carried,3,527,239,34,17.0,45.351044,14.773504,19.6
3,Carried,4,503,227,34,23.25,45.129225,19.6,182.0
4,Played,1,58,18,30,8.215909,31.034483,0.0,10.487245
5,Played,2,51,17,20,13.125,33.333333,10.487245,14.773504
6,Played,3,65,16,30,17.142857,24.615385,14.773504,19.6
7,Played,4,82,42,29,25.971429,51.219512,19.6,182.0
